# 📥 Análise Bronze — Diagnóstico de Qualidade dos Dados Brutos
**Pipeline Metrópole SP · Arquitetura Medallion**

> Esta camada armazena os dados **exatamente como chegam** das fontes.
> O objetivo deste notebook é **perfilar**, **diagnosticar** e **documentar** os problemas de qualidade.


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#070b14', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a3050',   'grid.color': '#1a3050',
    'text.color': '#d0e4ff',       'axes.labelcolor': '#7a9ab8',
    'xtick.color': '#4a6a8a',      'ytick.color': '#4a6a8a',
    'axes.titlecolor': '#00d4ff',  'axes.titlesize': 13,
    'axes.titleweight': 'bold',    'axes.grid': True,
    'figure.dpi': 120,
})
CYAN, GREEN, PURPLE = '#00d4ff', '#00ff88', '#7b2fff'
ORANGE, PINK, GOLD  = '#ff6b00', '#ff2d78', '#ffd700'
NEON = [CYAN, GREEN, PURPLE, ORANGE, PINK, GOLD]

# Detecta raiz do projeto
BASE = Path(os.environ.get('METRO_SP_BASE', Path.cwd()))
if not (BASE / 'data').exists():
    BASE = BASE.parent
BRONZE = BASE / 'data' / 'bronze'
SILVER = BASE / 'data' / 'silver'
GOLD   = BASE / 'data' / 'gold'
print(f"📂 Projeto: {BASE}")
print(f"   Bronze:  {BRONZE.exists()} | Silver: {SILVER.exists()} | Gold: {GOLD.exists()}")


In [ ]:
# ─── Carrega todas as fontes Bronze ──────────────────────────────────────────
arquivos = {
    'IoT Tráfego':    BRONZE / 'iot_traffic_raw.parquet',
    'Qualidade do Ar': BRONZE / 'air_quality_raw.parquet',
    'GPS Ônibus':     BRONZE / 'gps_bus_raw.parquet',
    'Ouvidoria':      BRONZE / 'ouvidoria_cdc_raw.parquet',
    'Meteorologia':   BRONZE / 'weather_inmet_raw.parquet',
}

dfs = {}
for nome, path in arquivos.items():
    if path.exists():
        dfs[nome] = pd.read_parquet(path)
        print(f"  ✅ {nome:<22} {len(dfs[nome]):>5,} registros  |  {len(dfs[nome].columns)} colunas")
    else:
        print(f"  ❌ {nome:<22} ARQUIVO NÃO ENCONTRADO — rode main.py primeiro")


In [ ]:
# ─── Perfil de Qualidade por Fonte ───────────────────────────────────────────
rows = []
for nome, df in dfs.items():
    rows.append({
        'Fonte': nome,
        'Registros': len(df),
        'Colunas': len(df.columns),
        'Nulos (%)': f"{df.isna().mean().mean()*100:.1f}%",
        'Duplicados': df.duplicated().sum(),
        'Tipos únicos': df.dtypes.nunique(),
    })

perfil = pd.DataFrame(rows).set_index('Fonte')
print("\n──────────────────────────────────────────────────────")
print("  PERFIL DE QUALIDADE — CAMADA BRONZE")
print("──────────────────────────────────────────────────────")
print(perfil.to_string())
print("──────────────────────────────────────────────────────")
perfil


In [ ]:
# ─── Mapa de Nulos por Fonte ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(dfs), figsize=(18, 5), facecolor='#070b14')
fig.suptitle('Mapa de Valores Nulos — Camada Bronze', fontsize=14,
             color=CYAN, fontweight='bold', y=1.02)

for ax, (nome, df) in zip(axes, dfs.items()):
    null_pct = df.isna().mean() * 100
    null_pct = null_pct[null_pct > 0] if (null_pct > 0).any() else null_pct.head(8)
    bars = ax.barh(null_pct.index, null_pct.values, color=ORANGE, alpha=0.85)
    ax.set_title(nome, fontsize=9, color=CYAN)
    ax.set_xlabel('Nulos (%)', fontsize=8)
    ax.set_xlim(0, 100)
    for bar, val in zip(bars, null_pct.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=7, color='#d0e4ff')
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ─── IoT Tráfego — Análise dos Sensores ───────────────────────────────────────
if 'IoT Tráfego' in dfs:
    df_tr = dfs['IoT Tráfego']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#070b14')
    fig.suptitle('IoT Tráfego — Problemas da Camada Bronze', color=CYAN, fontweight='bold')

    # Contagem de veículos (inclui -1 para sensores em manutenção)
    axes[0].hist(df_tr['contagem_veiculos'].dropna(), bins=40, color=CYAN, alpha=0.8, edgecolor='none')
    axes[0].axvline(-1, color=PINK, linestyle='--', linewidth=2, label='Sensor em manutenção (=-1)')
    axes[0].set_title('Contagem de Veículos (com anomalias)', fontsize=10)
    axes[0].legend(fontsize=8)

    # Bateria (%)
    axes[1].hist(df_tr['bateria_pct'].dropna(), bins=30, color=GREEN, alpha=0.8, edgecolor='none')
    axes[1].set_title('Nível de Bateria dos Sensores (%)', fontsize=10)

    # Velocidade (null quando contagem=0)
    null_vel = df_tr['velocidade_media_kmh'].isna().sum()
    ok_vel   = df_tr['velocidade_media_kmh'].notna().sum()
    axes[2].bar(['Com velocidade', 'Velocidade nula'], [ok_vel, null_vel],
                color=[CYAN, ORANGE], alpha=0.85)
    axes[2].set_title(f'Velocidade: {null_vel} nulos ({null_vel/len(df_tr):.1%})', fontsize=10)

    plt.tight_layout()
    plt.show()
    print(f"\n⚠  Sensores em manutenção (contagem=-1): {(df_tr['contagem_veiculos']==-1).sum()}")
    print(f"⚠  Velocidade nula:                       {null_vel} registros")
    print(f"⚠  Timestamps sem timezone explícito:      {len(df_tr)} registros (todos)")


In [ ]:
# ─── Qualidade do Ar — Problemas de Formato ────────────────────────────────────
if 'Qualidade do Ar' in dfs:
    df_ar = dfs['Qualidade do Ar']
    print("Tipos de dados originais (Bronze — tudo como object):")
    print(df_ar.dtypes.to_string())
    print(f"\nColunas de string com valores vazios '':")
    for col in df_ar.select_dtypes(include='object').columns:
        n_empty = (df_ar[col] == '').sum()
        if n_empty > 0:
            print(f"  {col:<12}: {n_empty:>4} vazios ({n_empty/len(df_ar):.1%})")


In [ ]:
# ─── GPS Ônibus — PII e Coordenadas ───────────────────────────────────────────
if 'GPS Ônibus' in dfs:
    df_gps = dfs['GPS Ônibus']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#070b14')
    fig.suptitle('GPS Ônibus — Problemas Bronze', color=CYAN, fontweight='bold')

    # Coordenadas 0,0 (bug de fora de serviço)
    coord_zero = ((df_gps['lat'] == 0) & (df_gps['lon'] == 0)).sum()
    coord_ok   = len(df_gps) - coord_zero
    axes[0].bar(['Coordenadas válidas', 'Coord (0,0) — bug'], [coord_ok, coord_zero],
                color=[GREEN, PINK], alpha=0.85)
    axes[0].set_title(f'Coordenadas inválidas: {coord_zero} ({coord_zero/len(df_gps):.1%})', fontsize=10)

    # Enum de lotação inconsistente
    axes[1].barh(df_gps['lotacao'].value_counts().index,
                 df_gps['lotacao'].value_counts().values,
                 color=[CYAN,PURPLE,GREEN,ORANGE,PINK,GOLD,'#888'][:len(df_gps['lotacao'].unique())],
                 alpha=0.85)
    axes[1].set_title('Enum de Lotação (PT/EN misturado)', fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f"\n🔒 LGPD: motorista_id exposto em {df_gps['motorista_id'].notna().sum():,} registros")
    print(f"   Exemplo: {df_gps['motorista_id'].iloc[0]}")


In [ ]:
# ─── Sumário de Problemas Identificados ───────────────────────────────────────
problemas = {
    'IoT Tráfego':     ['contagem_veiculos = -1 (manutenção)',
                         'timestamps sem timezone (BRT sem marcador)',
                         'velocidade_media_kmh nula quando contagem=0'],
    'Qualidade do Ar': ['datas em DD/MM/YYYY (não ISO)',
                         'campos ausentes como "" em vez de NaN',
                         'CO em ppm para estação CAC001 (deveria ser µg/m³)',
                         'MP2.5 negativo (sensor com defeito)'],
    'GPS Ônibus':      ['timestamp UNIX epoch sem fuso horário',
                         'enum lotação inconsistente (CHEIO, CHEIA, FULL)',
                         'coordenadas (0,0) para veículos fora de serviço',
                         'motorista_id exposto (PII — LGPD Art. 5)'],
    'Ouvidoria':       ['CPF e telefone em texto livre (PII — LGPD)',
                         'status PT/EN misturado (OPEN, CLOSED, ABERTO)',
                         '~12% sem coordenadas geográficas',
                         'abreviações inconsistentes de logradouro'],
    'Meteorologia':    ['todos os campos numéricos chegam como string',
                         'ausentes representados pela string "null"'],
}

print("=" * 65)
print("  DIAGNÓSTICO BRONZE — PROBLEMAS IDENTIFICADOS POR FONTE")
print("=" * 65)
total = 0
for fonte, issues in problemas.items():
    print(f"\n  📂 {fonte}")
    for issue in issues:
        print(f"      ⚠  {issue}")
        total += 1
print(f"\n  Total de problemas mapeados: {total}")
print("=" * 65)
print("\n→ Estes problemas são corrigidos na camada SILVER.")
